# Demo: attribute interference in text-to-image prompts (P40)

The full pipeline on a few representative prompts: factorial prompt → SD 1.5 (baseline) and SDXL on the same seed →
Qwen2.5-VL (8-bit) answers one question per constraint → labels and strict accuracy. It uses the same code and configs
as the main experiment (`attribute-interference generate`, `scripts/evaluate_labels.py`, `configs/main_*.yaml`).

Examples:
1. **Minimal pair, seed 1:** `a photo of a green horse` → `… and a black book`. Adding a second object breaks the color.
2. **Count and position, seed 2:** `a photo of three blue metal benches on the left side of the image`.

**Run order:** Runtime → T4 GPU → run all. The first cell installs packages and restarts the session; after that,
run all again. Generation and labeling take about 10 minutes, so run everything before the presentation.
All steps resume, so a rerun only shows the saved outputs.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/cv_project")
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/IzyumovMaxim/cv_project", str(REPO)], check=True)
os.chdir(REPO)
sys.path[:0] = [str(REPO / "src"), str(REPO / "scripts")]

MARKER = Path("/content/.demo_installed")
if not MARKER.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes==0.47.0"], check=True)
    MARKER.touch()
    print("Installed. Restarting the session; run all cells again.")
    os.kill(os.getpid(), 9)
print("Commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
GPU = torch.cuda.get_device_name(0)
print(GPU, round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

## 2. Prompts from the factorial set

The prompts come from the same generator as the main experiment: 20 objects × 5 binary constraints = 576 prompts.
Prompt IDs match the main runs, so our outputs can be compared with the reported images and human labels.

In [ ]:
import hashlib
import pandas as pd
from attribute_interference.prompts import build
from attribute_interference.storage import atomic_json, digest, load_config, source_hash, write_csv

CONFIGS = {"SD 1.5": "configs/main_sd15.yaml", "SDXL": "configs/main_sdxl.yaml"}
PROMPTS, PAIRS = build(load_config(CONFIGS["SD 1.5"]))
BY_ID = {p["prompt_id"]: p for p in PROMPTS}
print(f"{len(PROMPTS)} prompts, {len(PAIRS)} minimal pairs")

DEMO_JOBS = [
    ("e9e226eda3b2fff5", 1),  # a green horse
    ("a0870a216985d85a", 1),  # a green horse and a black book
    ("cdc2468937782175", 2),  # three blue metal benches on the left
]
pd.DataFrame([{"image_id": f"{pid}_s{seed}", "constraints": BY_ID[pid]["active_factors"],
               "prompt": BY_ID[pid]["prompt"]} for pid, seed in DEMO_JOBS])

## 3. Demo runs

`make_run` is `prepare` from `src/attribute_interference/prompts.py` restricted to the chosen jobs: the same
prompt rows, config and run identity checks, but 3 images instead of 2,304.

In [ ]:
DEMO_ROOT = Path("/content/demo_runs")


def make_run(name, config, jobs):
    root = DEMO_ROOT / name
    if (root / "run.json").exists():
        return root
    cfg = load_config(config)
    ids = {pid for pid, _ in jobs}
    root.mkdir(parents=True, exist_ok=True)
    write_csv(root / "prompts.csv", [p for p in PROMPTS if p["prompt_id"] in ids])
    write_csv(root / "pairs.csv", [p for p in PAIRS if p["base_id"] in ids and p["added_id"] in ids],
              ["base_id", "added_id", "added_factor"])
    write_csv(root / "jobs.csv", [{"image_id": f"{pid}_s{seed}", "prompt_id": pid, "seed": seed} for pid, seed in jobs])
    atomic_json(root / "run.json", {
        "config": cfg, "config_hash": digest(cfg), "source_hash": source_hash(),
        "prompt_count": len(ids), "job_count": len(jobs),
        "data_hashes": {n: hashlib.sha256((root / n).read_bytes()).hexdigest()
                        for n in ["prompts.csv", "pairs.csv", "jobs.csv"]}})
    return root


RUNS = {model: make_run(model.replace(" ", "").replace(".", "").lower(), cfg, DEMO_JOBS)
        for model, cfg in CONFIGS.items()}
RUNS

## 4. Generation

Same command as in the README. SD 1.5: about 7 s per image on a T4; SDXL: about 35 s per image (plus model download).

In [ ]:
!attribute-interference generate --config {CONFIGS["SD 1.5"]} --output {RUNS["SD 1.5"]}

In [ ]:
!attribute-interference generate --config {CONFIGS["SDXL"]} --output {RUNS["SDXL"]}

## 5. Labeling with Qwen2.5-VL (8-bit)

The labeling script of the main runs. OWLv2 also runs on prompts with count or position, as a comparison.
`--limit` only skips experiment-level metrics, which need the full run.

In [ ]:
!PYTHONPATH=src python scripts/evaluate_labels.py --config {CONFIGS["SD 1.5"]} --run {RUNS["SD 1.5"]} --limit 100

In [ ]:
!PYTHONPATH=src python scripts/evaluate_labels.py --config {CONFIGS["SDXL"]} --run {RUNS["SDXL"]} --limit 100

## 6. Model outputs

For each prompt: both images, every question Qwen was asked, its raw answer and the label.
A prompt is strictly correct only if the object and every requested constraint are correct.

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image

NAMES = {"object": "Object", "C": "Color", "N": "Count", "P": "Position", "M": "Material", "O2": "Second object"}


def qwen_record(model, image_id):
    return json.loads((RUNS[model] / "qwen_int8_records" / f"{image_id}.json").read_text())


def generation_record(model, image_id):
    return json.loads((RUNS[model] / "generation_records" / f"{image_id}.json").read_text())


def answers(model, image_id):
    rec = qwen_record(model, image_id)
    rows = {}
    for factor, raw in rec["raw"].items():
        label = rec["parsed"].get(factor)
        if rec["parsed"].get("object") is False and factor != "object":
            label = False
        rows[factor] = (raw["answer"].strip(), label)
    return rows


def mark(label):
    return {True: "✓", False: "✗"}.get(label, "?")


def strict(rows):
    labels = [label for _, label in rows.values()]
    return None if None in labels else all(labels)


def show(pid, seed):
    image_id = f"{pid}_s{seed}"
    p = BY_ID[pid]
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
    table = {}
    for ax, model in zip(axes, CONFIGS):
        gen = generation_record(model, image_id)
        ax.imshow(Image.open(RUNS[model] / gen["path"]))
        ax.set_axis_off()
        rows = answers(model, image_id)
        ax.set_title(f"{model}: strict {mark(strict(rows))}  ({gen['seconds']:.1f} s)", fontsize=11)
        for factor, (answer, label) in rows.items():
            table.setdefault(NAMES[factor], {})[f"{model} answer"] = answer
            table[NAMES[factor]][f"{model}"] = mark(label)
    fig.suptitle(f'"{p["prompt"]}", seed {seed}', fontsize=12)
    plt.tight_layout()
    plt.show()
    questions = {NAMES[f]: r["question"] for f, r in qwen_record("SD 1.5", image_id)["raw"].items()}
    df = pd.DataFrame(table).T
    df.insert(0, "question", pd.Series(questions))
    display(df)

In [ ]:
show("e9e226eda3b2fff5", 1)

In [ ]:
show("a0870a216985d85a", 1)

In [ ]:
show("cdc2468937782175", 2)

## 7. Checks against the main experiment

1. **Same images?** SHA-256 of each demo image against `results/image_hashes/`. SD 1.5 ran on a T4, so on a T4 the
   hashes should match. Our SDXL images for these prompts were made on an RTX 3090, so on a T4 they may differ slightly.
2. **Human labels.** Two of these SDXL images are in the 150-image human test sample (`notebooks/review/annotations_test150.csv`).
3. **OWLv2 vs Qwen counts** on the bench prompt.

In [ ]:
reference = {m: pd.read_csv(f"results/image_hashes/main_{m}.csv").set_index("image_id")
             for m in ["sd15", "sdxl"]}
rows = []
for model, key in [("SD 1.5", "sd15"), ("SDXL", "sdxl")]:
    for pid, seed in DEMO_JOBS:
        image_id = f"{pid}_s{seed}"
        ours = generation_record(model, image_id)["sha256"]
        ref = reference[key].loc[image_id]
        rows.append({"model": model, "image_id": image_id, "experiment GPU": ref["gpu"],
                     "identical to the experiment image": ours == ref["sha256"]})
print("Demo GPU:", GPU)
pd.DataFrame(rows)

In [ ]:
human = pd.read_csv("notebooks/review/annotations_test150.csv", sep=";", dtype=str).set_index("image_id")
rows = []
for pid, seed in DEMO_JOBS:
    image_id = f"{pid}_s{seed}"
    if image_id not in human.index:
        continue
    qwen = answers("SDXL", image_id)
    for factor in ["object"] + BY_ID[pid]["active_factors"].split("|"):
        value = str(human.loc[image_id, f"human_{factor}"]).strip().lower()
        rows.append({"image_id": image_id, "constraint": NAMES[factor],
                     "human": mark({"true": True, "false": False}.get(value)),
                     "Qwen (SDXL)": mark(qwen[factor][1])})
pd.DataFrame(rows)

In [ ]:
image_id = "cdc2468937782175_s2"
p = BY_ID["cdc2468937782175"]
rows = []
for model in CONFIGS:
    owl = json.loads((RUNS[model] / "owl_records" / f"{image_id}.json").read_text())
    qwen = qwen_record(model, image_id)["raw"]["N"]
    rows.append({"model": model, "requested": p["count"], "Qwen count": qwen["parsed_count"],
                 "OWLv2 boxes": len(owl["boxes"])})
pd.DataFrame(rows)

## 8. Results of the full experiment

2,304 images per model, Qwen labels. These numbers come from the files in `results/`, which the scripts in the README
reproduce without a GPU.

In [ ]:
acc = pd.read_csv("results/report_tables/accuracy.csv").set_index("metric")
acc.index = [NAMES.get(m, "Strict") for m in acc.index]
fig, ax = plt.subplots(figsize=(8, 3.6))
x = range(len(acc))
for i, (col, name, color) in enumerate([("sd15", "SD 1.5", "#2a78d6"), ("sdxl", "SDXL", "#eb6834")]):
    bars = ax.bar([v + (i - 0.5) * 0.38 for v in x], acc[col] * 100, width=0.36, label=name, color=color)
    ax.bar_label(bars, fmt="%.0f", fontsize=8, color="#444")
ax.set_xticks(list(x), acc.index)
ax.set_ylabel("accuracy, %")
ax.set_ylim(0, 105)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
ax.legend(frameon=False)
ax.set_title("Attribute and strict accuracy, 2,304 images per model")
plt.tight_layout()
plt.show()

In [ ]:
inter = pd.read_csv("results/report_tables/interference.csv")
top = inter[inter["mcnemar_p_holm"] < 0.05].sort_values("delta").head(8)
top = top.assign(checked=top["checked"].map(NAMES), added=top["added"].map(NAMES),
                 delta=(top["delta"] * 100).round(1), holm_p=top["mcnemar_p_holm"].map("{:.1e}".format))
print("Largest drops when one constraint is added (same seed, minimal pairs), percentage points:")
top[["model", "checked", "added", "pairs", "delta", "holm_p"]].reset_index(drop=True)

In [ ]:
by_k = pd.read_csv("results/report_tables/strict_by_constraints.csv")
by_k = by_k[by_k["constraints"] > 0].pivot(index="constraints", columns="model", values="strict_accuracy")
(by_k * 100).round(1).rename(columns={"sd15": "SD 1.5, %", "sdxl": "SDXL, %"})

## 9. Optional: your own prompt

Pick an object and constraints; the values (color, count, side, …) are fixed per object by the factorial generator.
This runs SD 1.5 and Qwen, about 2–3 minutes on a T4 because Qwen is loaded again.

In [ ]:
OBJECT = "cup"            # cup bowl vase chair bottle spoon clock table box lamp ball bucket tray stool bench toy car dog cat bird horse
CONSTRAINTS = ["C", "O2"]  # any of C N P M O2 (M not for animals)
SEED = 0

order = ["C", "N", "P", "M", "O2"]
key = "|".join(f for f in order if f in CONSTRAINTS)
pid = next(p["prompt_id"] for p in PROMPTS if p["object"] == OBJECT and p["active_factors"] == key)
print(BY_ID[pid]["prompt"])
LIVE = make_run(f"live_{pid}_s{SEED}", CONFIGS["SD 1.5"], [(pid, SEED)])

In [ ]:
!attribute-interference generate --config {CONFIGS["SD 1.5"]} --output {LIVE}
!PYTHONPATH=src python scripts/evaluate_labels.py --config {CONFIGS["SD 1.5"]} --run {LIVE} --limit 100
RUNS["live"] = LIVE
image_id = f"{pid}_s{SEED}"
rows = answers("live", image_id)
plt.figure(figsize=(4, 4))
plt.imshow(Image.open(LIVE / generation_record("live", image_id)["path"]))
plt.axis("off")
plt.title(f"SD 1.5: strict {mark(strict(rows))}")
plt.show()
pd.DataFrame([{"constraint": NAMES[f], "answer": a, "label": mark(l)} for f, (a, l) in rows.items()])